# Step 6: Train the reaction-conditions model (Model 2, Colab GPU)

Runs `scripts/train_conditions_model.py`, a full fine-tune of a plain,
non-chemical `t5-small` on the freshly built `data/v2_ord_train/conditions_train.jsonl`
to predict solvent/catalyst/temperature/yield from a product+reactants pair.

**Colab session budget: ~3h/day.** Checkpoints are written to Google Drive; re-run
this notebook on a later day to resume from the last checkpoint.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

`data/v2_ord_train/` is gitignored (large, derived) -- regenerate it deterministically
here (fixed seed, excludes the committed `data/v2_ord_eval_targets.json` by
construction). Only needs to run once per Colab session.

In [ ]:
import os

if not os.path.exists("data/v2_ord_train/conditions_train.jsonl"):
    !python scripts/build_train_data_ord.py --pool-count 60000 --seed 42

In [ ]:
output_dir = "/content/drive/MyDrive/retro-planner-checkpoints/model2_conditions"  # @param {type:"string"}
time_budget_minutes = 165  # @param {type:"number"}
base_model = "t5-small"  # @param ["t5-small", "t5-base"]

In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"

!python scripts/train_conditions_model.py \
    --base-model "{base_model}" \
    --output-dir "{output_dir}" \
    --time-budget-minutes {time_budget_minutes} \
    > "{log_path}" 2>&1
print(f"Done (or paused at time budget). Log: {log_path}")

Training output is redirected to `train.log` in `output_dir` (on Drive) instead of
printing here, to avoid the notebook's output growing large enough to make the browser
tab unresponsive on a long run. Open `train.log` in Google Drive's own web preview to
check progress -- that works independently of the Colab kernel, which stays busy
(blocked) running the cell above.

Re-run the cell above (same `output_dir`) on the next day's Colab session to
continue training. Once finished, evaluate against
`data/v2_ord_train/conditions_test.jsonl` (held out, never used in training).